# Generalization tests

Source: archived main notebook, cells 78–104. Both-reactant holdout excludes reactions with only one held-out reactant.


In [7]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [8]:
import numpy as np
import pandas as pd
from _modeling_common import (PHYSORG_FEATURE_COUNT, TARGET_COLUMNS,
    broad_training_ids, evaluate_targets, fit_delta_g_feature,
    load_modeling_data, metrics, new_catboost)

data_Ga, X_Ga, data_G, X_G = load_modeling_data(DATA_DIR, DESCRIPTOR_MAP)
Y_DD = data_Ga["Diene_Distort"].to_numpy()
Y_ED = data_Ga["Ene_Distort"].to_numpy()
Y_IN = data_Ga["Interaction"].to_numpy()
Y_Ga = data_Ga["deltaGa(Solvent)"].to_numpy()
Y_G = data_Ga["deltaG"].to_numpy()
Y_G_data_G = data_G["deltaG"].to_numpy()
print(f"Aligned active-energy rows: {len(data_Ga):,}; broad rows: {len(data_G):,}")

100%|██████████| 50161/50161 [00:07<00:00, 6815.74it/s]


Aligned active-energy rows: 8,273; broad rows: 50,161


In [9]:
from Code import model_train

VARIANTS = ["PhysOrg", "PhysOrg + SVO",
            "PhysOrg + SVO + predicted reaction energy (active only)",
            "PhysOrg + SVO + predicted reaction energy (broad)"]

def reactant_split(frame, seed, holdout):
    """Mirror OOD_train_test in 3_model_train.ipynb."""
    np.random.seed(seed)
    diene_indexs = np.sort(np.unique(frame["Diene_Index"].to_numpy()))
    ene_indexs = np.sort(np.unique(frame["Ene_Index"].to_numpy()))
    np.random.shuffle(diene_indexs)
    np.random.shuffle(ene_indexs)
    removed_dienes = diene_indexs[:len(diene_indexs) // 10]
    removed_enes = ene_indexs[:len(ene_indexs) // 10]

    if holdout == "both":
        train_ids = [row_id for row_id, row in frame.iterrows()
                     if row["Diene_Index"] not in removed_dienes
                     and row["Ene_Index"] not in removed_enes]
        test_ids = [row_id for row_id, row in frame.iterrows()
                    if row["Diene_Index"] in removed_dienes
                    and row["Ene_Index"] in removed_enes]
    elif holdout == "diene":
        train_ids = [row_id for row_id, row in frame.iterrows()
                     if row["Diene_Index"] not in removed_dienes]
        test_ids = [row_id for row_id, row in frame.iterrows()
                    if row["Diene_Index"] in removed_dienes]
    elif holdout == "ene":
        train_ids = [row_id for row_id, row in frame.iterrows()
                     if row["Ene_Index"] not in removed_enes]
        test_ids = [row_id for row_id, row in frame.iterrows()
                    if row["Ene_Index"] in removed_enes]
    else:
        raise ValueError(holdout)
    return train_ids, test_ids
def evaluate_splits(splits, split_type):
    results = []
    for split_id, (train_ids, test_ids) in enumerate(splits):
        if len(train_ids) < 2 or len(test_ids) < 2:
            raise ValueError(f"Split {split_id} has too few rows")
        for variant in VARIANTS:
            if variant == "PhysOrg":
                features = X_Ga[:, :PHYSORG_FEATURE_COUNT]
            elif variant == "PhysOrg + SVO":
                features = X_Ga
            else:
                source = "active" if variant.endswith("(active only)") else "broad"
                pred_g = fit_delta_g_feature(data_G, X_G, data_Ga, X_Ga,
                                             train_ids, test_ids, source=source)
                features = np.column_stack([X_Ga, pred_g])
            rows, _ = evaluate_targets(data_Ga, features, train_ids, test_ids)
            results.extend({"split_type": split_type, "split": split_id,
                            "n_train": len(train_ids), "n_test": len(test_ids),
                            "variant": variant, **row} for row in rows)
    return pd.DataFrame(results)

In [10]:
ood_results = []
for holdout in ["diene", "ene", "both"]:
    splits = [reactant_split(data_Ga, seed, holdout) for seed in [0, 3, 6]]
    ood_results.append(evaluate_splits(splits, f"{holdout} reactant OOD"))
ood_results = pd.concat(ood_results, ignore_index=True)
display(ood_results.groupby(["split_type", "variant", "target"])[["r2", "mae"]].mean())

r2  \
split_type         variant                                            target                                     
both reactant OOD  PhysOrg                                            Aqueous activation free energy  0.587607   
                                                                      Diene distortion                0.619068   
                                                                      Ene distortion                  0.486310   
                                                                      Interaction                     0.509077   
                   PhysOrg + SVO                                      Aqueous activation free energy  0.663823   
                                                                      Diene distortion                0.674330   
                                                                      Ene distortion                  0.548385   
                                                                      Interaction                     0.561620   
                   PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  0.658501   
                                                                      Diene distortion                0.695517   
                                                                      Ene distortion                  0.571470   
                                                                      Interaction                     0.575975   
                   PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  0.768071   
                                                                      Diene distortion                0.762532   
                                                                      Ene distortion                  0.712426   
                                                                      Interaction                     0.625408   
diene reactant OOD PhysOrg                                            Aqueous activation free energy  0.608663   
                                                                      Diene distortion                0.612269   
                                                                      Ene distortion                  0.575228   
                                                                      Interaction                     0.639907   
                   PhysOrg + SVO                                      Aqueous activation free energy  0.663932   
                                                                      Diene distortion                0.651625   
                                                                      Ene distortion                  0.626465   
                                                                      Interaction                     0.658265   
                   PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  0.676765   
                                                                      Diene distortion                0.680881   
                                                                      Ene distortion                  0.629725   
                                                                      Interaction                     0.660071   
                   PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  0.782583   
                                                                      Diene distortion                0.736668   
                                                                      Ene distortion                  0.716457   
                                                                      Interaction                     0.672654   
ene reactant OOD   PhysOrg                                            Aqueous activation free energy  0.783999   
                                                                      Diene distortion                0.849187   
    

## Grouped reaction folds


In [12]:
grouped_folds = list(model_train.special_k_fold(data_Ga, 5, 0, True))
grouped_results = evaluate_splits(grouped_folds, "grouped reaction fold")
display(grouped_results.groupby(["variant", "target"])[["r2", "mae"]].mean())

r2  \
variant                                            target                                     
PhysOrg                                            Aqueous activation free energy  0.828521   
                                                   Diene distortion                0.875141   
                                                   Ene distortion                  0.676962   
                                                   Interaction                     0.663948   
PhysOrg + SVO                                      Aqueous activation free energy  0.844518   
                                                   Diene distortion                0.883623   
                                                   Ene distortion                  0.724742   
                                                   Interaction                     0.697232   
PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  0.841856   
                                                   Diene distortion                0.885681   
                                                   Ene distortion                  0.723704   
                                                   Interaction                     0.699314   
PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  0.865026   
                                                   Diene distortion                0.896609   
                                                   Ene distortion                  0.742363   
                                                   Interaction                     0.699257   

                                                                                        mae  
variant                                            target                                    
PhysOrg                                            Aqueous activation free energy  2.391388  
                                                   Diene distortion                1.560232  
                                                   Ene distortion                  1.598431  
                                                   Interaction                     2.049866  
PhysOrg + SVO                                      Aqueous activation free energy  2.234229  
                                                   Diene distortion                1.488493  
                                                   Ene distortion                  1.470319  
                                                   Interaction                     1.928824  
PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  2.242678  
                                                   Diene distortion                1.470013  
                                                   Ene distortion                  1.469944  
                                                   Interaction                     1.921680  
PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  2.113967  
                                                   Diene distortion                1.407054  
                                                   Ene distortion                  1.420742  
                                                   Interaction                     1.918079

## Barrier-ordered diagnostic

Test folds use contiguous ranges of the measured barrier. This is a diagnostic stress test.


In [ ]:
sorted_ids = np.argsort(Y_Ga)
ordered_folds = []
for test_ids in np.array_split(sorted_ids, 5):
    test_ids = np.sort(test_ids)
    train_ids = np.setdiff1d(np.arange(len(Y_Ga)), test_ids)
    ordered_folds.append((train_ids, test_ids))
ordered_results = evaluate_splits(ordered_folds, "barrier ordered")
windows = pd.DataFrame([{"fold": fold, "n_test": len(test_ids),
                         "min_barrier": Y_Ga[test_ids].min(),
                         "max_barrier": Y_Ga[test_ids].max()}
                        for fold, (_, test_ids) in enumerate(ordered_folds)])
display(windows)
display(ordered_results.groupby(["variant", "target"])[["r2", "mae"]].mean())

,fold,n_test,min_barrier,max_barrier
0,0,1655,2.308818,32.525671
1,1,1655,32.527948,35.941285
2,2,1655,35.941405,39.702527
3,3,1654,39.704082,45.336040
4,4,1654,45.354884,65.940577


r2  \
variant                                            target                                     
PhysOrg                                            Aqueous activation free energy -4.477112   
                                                   Diene distortion                0.592329   
                                                   Ene distortion                  0.390263   
                                                   Interaction                     0.592145   
PhysOrg + SVO                                      Aqueous activation free energy -3.527554   
                                                   Diene distortion                0.613207   
                                                   Ene distortion                  0.525091   
                                                   Interaction                     0.649644   
PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy -3.515951   
                                                   Diene distortion                0.625142   
                                                   Ene distortion                  0.532305   
                                                   Interaction                     0.647435   
PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy -3.235472   
                                                   Diene distortion                0.680297   
                                                   Ene distortion                  0.586123   
                                                   Interaction                     0.650023   

                                                                                        mae  
variant                                            target                                    
PhysOrg                                            Aqueous activation free energy  4.066383  
                                                   Diene distortion                1.883647  
                                                   Ene distortion                  1.969465  
                                                   Interaction                     2.149563  
PhysOrg + SVO                                      Aqueous activation free energy  3.766403  
                                                   Diene distortion                1.826447  
                                                   Ene distortion                  1.729963  
                                                   Interaction                     2.005882  
PhysOrg + SVO + predicted reaction energy (acti... Aqueous activation free energy  3.752232  
                                                   Diene distortion                1.789562  
                                                   Ene distortion                  1.709255  
                                                   Interaction                     2.018640  
PhysOrg + SVO + predicted reaction energy (broad)  Aqueous activation free energy  3.574910  
                                                   Diene distortion                1.659194  
                                                   Ene distortion                  1.605343  
                                                   Interaction                     2.011171